
### pymc3による非線形関数のベイズ回帰



第一原理量子モンテカルロ法を行うと電子相関を正しく取り入れた全エネルギーを
高精度で求めることが可能であるが、計算誤差が必ず含まれる。
線形回帰となるMorse potentialを仮定してベイズ回帰を行う。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

np.random.seed(11)


def Morsepotential(x, p):
    """Morse potential

    Args:
        x (np.array): descriptor
        p (list): Morse Potential parameter

    Returns:
        np.array: target values
    """
    """
    Morse potential
    """
    De = p[0]
    a = p[1]
    rc = p[2]
    return De * ((1-np.exp(-a*(x-rc)))**2 - 1)


def make_Xy(w_ans=[1.0, 0.5, 1.5], Ysigma=0.05):
    """make X and y
    YはMorse potentialにnoiseを加えて観測値とする。

    Args:
        w_ans (list, optional): Morse potential parameter. Defaults to [1.0, 0.5, 1.5].
        Ysigma (float, optional): noise to add. Defaults to 0.05.

    Returns:
        np.array: descriptor
        np.array: observed target value
        np.array: true target value
        np.array: noise for descriptor
    """
    X = np.linspace(0.2, 10, 20)
    #X = np.logspace(-1,1,20)
    Y0 = Morsepotential(X, w_ans)
    T = np.random.normal(loc=Y0, scale=Ysigma, size=X.shape[0])
    yerr = []
    for i in range(X.shape[0]):
        yerr.append(Ysigma)
    yerr = np.array(yerr)

    plt.figure()
    plt.plot(X, Y0, "--")
    plt.errorbar(X, T, yerr=3*yerr, fmt="o")
    plt.title("y +- 3*sigma")
    plt.xlabel("r")
    plt.ylabel("E(r)")
    plt.show()
    return X, T, Y0, yerr


g_X, g_T, g_Y0, g_Yerr = make_Xy()


Deが0.4eV程度の場合に、DMCで用いる波動関数に配置間相互作用を入れないと精度が上がらず上図のようになる。
この「実験」データからMorse potentialを仮定してポテンシャルパラメタ求めます。

モデルの定義を行います。

In [ ]:
"""
pymc modelの作成
"""

import pymc as pm


def make_model(X, T):

    basic_model = pm.Model()

    with basic_model:
        De = pm.Normal('De', mu=1.2, sd=0.1)
        a = pm.Normal('a', mu=0.55, sd=0.1)
        rc = pm.Normal('rc', mu=1.6, sd=0.3)

        sigma = pm.HalfNormal('sigma', sd=1)

        # 関数式をY0と同じにすること
        mu = De * ((1-np.exp(-a*(X-rc)))**2 - 1)
        Y_exp = pm.Normal('Y_exp', mu=mu, sd=sigma, observed=T)
        #Y_exp = pm.Normal('Y_exp', mu=mu, sd=Yerr, observed=T)
    return basic_model

import numpy as np
import pymc as pm


def make_model(X, T):
    X = np.asarray(X, dtype=float)
    T = np.asarray(T, dtype=float)

    with pm.Model() as basic_model:
        De = pm.Normal("De", mu=1.2, sigma=0.1)
        a  = pm.Normal("a",  mu=0.55, sigma=0.1)
        rc = pm.Normal("rc", mu=1.6, sigma=0.3)

        sigma = pm.HalfNormal("sigma", sigma=1.0)

        mu = De * ((1 - pm.math.exp(-a * (X - rc)))**2 - 1)

        Y_exp = pm.Normal(
            "Y_exp",
            mu=mu,
            sigma=sigma,
            observed=T
        )

    return basic_model

g_basic_model = make_model(g_X, g_T)


grahvizがあるとモデルの図示ができます。

In [ ]:
# pm.model_to_graphviz(basic_model)
print("comment out if you have it")


In [ ]:
import pickle
import os
g_filename_trace = "nonlinear_model_Morse.pickle"
if not os.path.isfile(g_filename_trace):
    with g_basic_model:
        n_pm_sample = 20000
        g_trace = pm.sample(n_pm_sample)
        # it runs n_sample x jobs
    with open(g_filename_trace,"wb") as _f:
        pickle.dump(g_trace, _f)
    # virtualbox上のububutu,4core,minicondaでは約40秒で終了します。
    # 一方、Anacondaではかなり時間がかかりました。
else:
    with open(g_filename_trace,"rb") as _f:
        g_trace = pickle.load(_f)

In [ ]:
"""
それぞれの変数変数毎の分布
"""
import arviz_plots as azp

#pm.traceplot(g_trace)
azp.plot_trace(g_trace)

In [ ]:
import pandas as pd
import seaborn as sns


def show_var_as_df(trace, var=["De", "a", "rc"]):
    df = pd.DataFrame()

    for x in var:
        df[x] = trace.posterior[x].values.flatten()

    return df
    

show_var_as_df(g_trace,)


In [ ]:
import random
import seaborn as sns
#sns.pairplot(df, diag_kind="kde" ,markers="")


In [ ]:
import matplotlib.pyplot as plt


def plot_distribution1D(trace, var):
    """plot posterior distribution"""

    # posterior sample取得
    a0 = trace.posterior[var].values.flatten()

    plt.figure(figsize=(5, 4))

    plt.hist(a0, bins=80)

    plt.xlabel(var)
    plt.ylabel("count")

    plt.show()


plot_distribution1D(g_trace, "De")
plot_distribution1D(g_trace, "a")
plot_distribution1D(g_trace, "rc")


In [ ]:
# seaborn の 非対角項がみにくいので別途kde plotにする。

import random
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt


def plot_distribution2D(trace, var):
    """
    posterior sample を2D KDE表示

    Args:
        trace : InferenceData / DataTree
        var : ["x", "y"]
    """

    # posterior samples取得
    a0 = trace.posterior[var[0]].values.flatten()
    a1 = trace.posterior[var[1]].values.flatten()

    # sample数
    n = min(2000, len(a0))

    # random sampling
    idx = np.random.choice(len(a0), size=n, replace=False)

    a0small = a0[idx]
    a1small = a1[idx]

    plt.figure(figsize=(5, 5))

    # KDE contour
    sns.kdeplot(
        x=a0small,
        y=a1small,
        fill=True
    )

    plt.title(f"{n} samples")

    plt.xlabel(var[0])
    plt.ylabel(var[1])

    plt.show()

plot_distribution2D(g_trace, ["De", "a"])
plot_distribution2D(g_trace, ["De", "rc"])
plot_distribution2D(g_trace, ["a", "rc"])


本来はmultivariate distributionです。
sample数が増えると綺麗な分布になります。


以下で各変数毎fitした結果をdata frameとして取り出す。
gaussainで無い関数もgussianでfitします。


In [ ]:
import arviz as az
# g_df = pm.summary(g_trace)
g_df = az.summary(g_trace).round(3)
g_df


最適値を取り出す。

In [ ]:

g_df["mean"]["De"], g_df["mean"]["a"], g_df["mean"]["rc"]


実験値と予測値を図示します。

In [ ]:
from numpy.random import multivariate_normal
from sklearn.mixture import GaussianMixture
import os

def show_curves(trace, var, X, T, Y0, Yerr):
    """plot y and y^predict +- sigma curves

    Args:
        trace (pm.sample): a list of sampled variables
        var (list): a list of variables
        X (np.array): samples descriptors
        T (np.array): observed target values
        Y0 (np.array): true target values        
        Yerr ([type]): errors of target values
    """
    plt.figure()
    # print("X.shape",X.shape,Y.shape,Y0.shape,Yerr)
    plt.plot(X, Y0, label="Y0", linewidth=1, color="blue")
    plt.errorbar(X, T, yerr=Yerr, fmt="o-",
                 label="T", linewidth=1, color="blue")

    # mc過程の取得
    #a0 = trace.posterior[var[0]]
    #a1 = trace.posterior[var[1]]
    #a2 = trace.posterior[var[2]]
    # と取り出せる。
    
    w_mc = []
    for var1 in var:
        print("add", var1)
        # w_mc.append(trace.get_values(var1))
        w_mc.append(trace.posterior[var1].values.flatten())        
        
    w_mc = np.array(w_mc).T

    # mean and covariance fit
    print("find means and covariances with multivariate gaussian")
    cls = GaussianMixture(n_components=1)
    cls.fit(w_mc)
    print("means", cls.means_)
    print("covariances")
    print(cls.covariances_)

    #  (cls.means_[0], cls.covariances_[0])で与えられるguassian分布からn個wを引く。
    n = 200
    w_rand = multivariate_normal(cls.means_[0], cls.covariances_[0], size=n)
    for i, w_rand1 in enumerate(w_rand):
        Y_rand = Morsepotential(X, w_rand1)
        plt.plot(X, Y_rand, "-", color="red", alpha=0.01)

    plt.legend()
    plt.xlabel("r")
    plt.ylabel("E")
    os.makedirs("image_executed", exist_ok=True)
    plt.savefig("image_executed/Morsepotential_MCMC_fit.png")

    plt.show()

show_curves(g_trace, ["De", "a", "rc"], g_X, g_T, g_Y0, g_Yerr)


In [ ]:
def plot_Yopt(X, Y, Yerr, df):
    """plot Y vs Y^predict

    Args:
        X (np.array): descriptor
        Y (np.array): observed target values
        Yerr (np.array): assumed error of target values
        df (pd.DataFrame): data
    """
    # 再評価
    Yp = Morsepotential(
        X, [df["mean"]["De"], df["mean"]["a"], df["mean"]["rc"]])

    # y vs yp
    plt.figure(figsize=(5, 5))
    plt.errorbar(Y, Yp, xerr=Yerr, yerr=df["mean"]["sigma"], fmt="o")
    # plt.legend()
    # plt.ylim((0,5))
    plt.xlabel("Y_exp with the mean parameters")
    plt.ylabel("Y(opt parameter)")
    y1, y2 = Y.min(), Y.max()
    plt.plot([y1, y2], [y1, y2], "--")
    plt.show()

def plot_Yopt(X, Y, Yerr, df):
    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    Yerr = np.asarray(Yerr, dtype=float)

    p = [
        float(df.loc["De", "mean"]),
        float(df.loc["a", "mean"]),
        float(df.loc["rc", "mean"]),
    ]

    Yp = Morsepotential(X, p)

    plt.figure(figsize=(5, 5))
    plt.errorbar(Y, Yp, xerr=Yerr, fmt="o")

    y1 = min(Y.min(), Yp.min())
    y2 = max(Y.max(), Yp.max())

    plt.plot([y1, y2], [y1, y2], "--")
    plt.xlabel("observed")
    plt.ylabel("predicted")
    plt.show()

g_X = np.asarray(g_X, dtype=float)
g_T = np.asarray(g_T, dtype=float)
g_Yerr = np.asarray(g_Yerr, dtype=float)

plot_Yopt(g_X, g_T, g_Yerr, g_df)


分布最大値の値だけも見ておきます。

In [ ]:
g_map_estimate = pm.find_MAP(model=g_basic_model)
print("map_estimate", g_map_estimate)
g_df



priorの設定によっては他の解にtrapされることもあります。

#### 以下はファイルがあると実行できる。

失敗例をサンプルデータとして保存してあります。
失敗があれば"data/morsepotential_sample.pickle"を置きます。


In [ ]:
!pwd


In [ ]:
import pickle
g_filename = "data/morsepotential_sample.pickle"

# データ作成用 routine


def save_trace(filename, trace):
    with open(filename, "wb") as f:
        pickle.dump(trace, f)


In [ ]:
import os
# データ読み込み用 routine

def get_trace(filename):
    print("try loading", filename)
    if os.path.exists(filename):
        with open(filename, "rb") as f:
            trace_sample = pickle.load(f)
    else:
        print("file not found")
        raise
    return trace


g_trace_sample = get_trace(g_filename)

pm.traceplot(g_trace_sample)
pm.summary(g_trace_sample)
pm.autocorrplot(g_trace_sample)
print("done")


自己相関は問題ないようですが、
sigmaが0.35付近にpeakを持つ異なるモンテカルロchainがあることに気づきます。
pm.summary(trace_sample)によるとおかしい（すべてのchainが同じ値に収束していない）ことがすでにわかるのですが、
そのデータだけを取り出し何が起きているのか見てみます。

全体を見て、どの範囲か特定します。

In [ ]:
def show_trace(trace):
    """show trace

    Args:
        trace (pm.sample): pm trace
    """
    a0 = trace.get_values("De")

    plt.figure()
    plt.plot(range(a0.shape[0]), a0)
    plt.title("show all")
    plt.show()

    i = 0
    for i in range(len(trace.chains)):
        i = 2000*i
        a0cut = a0[i:i+2000]
        plt.title("range {} {}".format(i, i+2000))
        plt.plot(range(a0cut.shape[0]), a0cut)
        plt.show()


show_trace(g_trace_sample)


[6000:8000]のデータが異なることが確認できました。以下はそのデータを用いて解を見ます。

In [ ]:
from numpy.random import multivariate_normal
from sklearn.mixture import GaussianMixture


def cutdata_fit_gaussian(trace, var, X, Y, Y0, Yerr):
    """use the strange data and fit the distribution with gaussian

    Args:
        trace (pm.sample): a list of sampled variables
        var (list): a list of variables
        X (np.array): descriptor
        Y (np.array): observed target values
        Y0 (np.array): true target values
        Yerr (np.array): assumed error of target values
    """
    print("cut 2000:4000, fit with gaussian and draw N samples")
    plt.figure()
    # print("X.shape",X.shape,Y.shape,Y0.shape,Yerr)
    plt.plot(X, Y0, label="Y0", linewidth=1)
    plt.errorbar(X, Y, yerr=3*Yerr, fmt="o-", label="Y0+3sigma", linewidth=1)

    # 2000:4000のデータの取り出し。
    i = 6000
    w_mc = []
    for var1 in var:
        print("add", var1)
        w_mc.append(trace.get_values(var1)[i:i+2000])
    w_mc = np.array(w_mc).T

    # mean and covariance fit
    print("find means and covariances with multivariate gaussian")
    cls = GaussianMixture(n_components=1)
    cls.fit(w_mc)
    print("means", cls.means_)
    print("covariances")
    print(cls.covariances_)

    #  guassian分布からn個wを引く。
    n = 10
    w_rand = multivariate_normal(cls.means_[0], cls.covariances_[0], size=n)
    print("w", w_rand)
    for i, w_rand1 in enumerate(w_rand):
        Y_rand = Morsepotential(X, w_rand1)
        plt.plot(X, Y_rand, "--")  # ,label="predict{}".format(i),linewidth=1)

    plt.legend()
    plt.show()


cutdata_fit_gaussian(g_trace_sample, ["De", "a", "rc"], g_X, g_T, g_Y0, g_Yerr)


上で示された解にtrapされていました。

aは
$$
a = \sqrt{ k_e / D_e }
$$
という物理的意味があります。ここで$k_e$ はr_eでの最小点でのforce constantです。
aがマイナスなのは物理的に考えておかしな解です。
また、sigmaももうひとつの解に比べて大きな値でした。
priorによってはこのような解になることがあります。（また、モンテカルロ法として収束していませんのでモンテカルロ法の適用としても誤っています。）

#### 参考文献

マルコフチェインモンテカルロの収束の診断方法は例えば以下がある。

1. http://web.sfc.keio.ac.jp/~maunz/BS14/BS14-11.pdf

例えば、Rhat ($\hat R$) はGelman-Rubinの診断方法による。



関連研究として以下のようなベイズ推定を利用したパラメタフィット研究がある。

1. model Hamiltonianからパラメタ分布を見つけ出す。
Hikaru Takenaka, Kenji Nagata, Takashi Mizokawa, and Masato Okada, https://journals.jps.jp/doi/10.7566/JPSJ.85.124003
2. noisy complex spectraからなるべく少ないピーク数と位置を同定する。
Satoru Tokuda, Kenji Nagata, and Masato Okada,https://journals.jps.jp/doi/10.7566/JPSJ.86.024001
